# 🚚 Optimizador de Costes Logísticos
**Versión:** 1.0  
**Descripción:** Calcula y optimiza el coste total de transporte + almacenaje para 5 tipos de producto.  
Modifica la sección **CONFIGURACIÓN** con tus datos reales.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
print('Librerías cargadas ✓')

---
## ⚙️ SECCIÓN 1 — CONFIGURACIÓN DE PRODUCTOS
> **Modifica aquí** los 5 productos con sus datos reales cuando los tengas.

In [ ]:
# ============================================================
#  PRODUCTOS  (largo cm, ancho cm, alto cm, peso kg, uds/caja)
# ============================================================
PRODUCTOS = {
    'Producto A': {'largo_cm': 40, 'ancho_cm': 30, 'alto_cm': 25, 'peso_kg': 5.0,  'uds_por_caja': 12},
    'Producto B': {'largo_cm': 60, 'ancho_cm': 40, 'alto_cm': 35, 'peso_kg': 12.0, 'uds_por_caja': 6},
    'Producto C': {'largo_cm': 20, 'ancho_cm': 20, 'alto_cm': 15, 'peso_kg': 2.5,  'uds_por_caja': 24},
    'Producto D': {'largo_cm': 80, 'ancho_cm': 60, 'alto_cm': 50, 'peso_kg': 20.0, 'uds_por_caja': 4},
    'Producto E': {'largo_cm': 30, 'ancho_cm': 25, 'alto_cm': 20, 'peso_kg': 3.5,  'uds_por_caja': 18},
}

# Dimensiones estándar de un palé (Europalet)
PALE_LARGO_CM  = 120
PALE_ANCHO_CM  = 80
PALE_ALTO_MAX_CM = 170   # altura máxima de carga
PALE_PESO_MAX_KG = 1000  # peso máximo por palé

# ============================================================
#  ESCENARIOS DE PEDIDO A EVALUAR (número de cajas por envío)
# ============================================================
ESCENARIOS_CAJAS = [10, 50, 100, 250, 500, 1000, 2000, 5000, 10000]

# ============================================================
#  DESTINOS
# ============================================================
# Provincia destino (para tarifa por palé) — 'peninsula' o 'baleares'
DESTINO_DEFAULT = 'peninsula'

print('Configuración de productos cargada ✓')
pd.DataFrame(PRODUCTOS).T

---
## 💶 SECCIÓN 2 — TARIFAS
> Modifica aquí los precios cuando tengas las tarifas exactas del proveedor.

In [ ]:
# ============================================================
#  TRANSPORTE BÁSICO POR PESO  (hasta 1000 kg)
#  Formato: (peso_max_kg, precio_peninsula, precio_baleares)
# ============================================================
TRANSPORTE_PESO = [
    #  kg_max   peninsula   baleares
    (    5,      8.50,       12.00),
    (   10,     11.00,       15.50),
    (   20,     15.00,       21.00),
    (   30,     18.50,       26.00),
    (   40,     22.00,       30.50),
    (   50,     25.50,       35.00),
    (   60,     28.00,       38.50),
    (   70,     30.50,       42.00),
    (   80,     33.00,       45.50),
    (  100,     37.00,       50.00),
    (  150,     48.00,       64.00),
    (  200,     58.00,       78.00),
    (  300,     75.00,      100.00),
    (  400,     90.00,      120.00),
    (  500,    105.00,      140.00),
    (  600,    118.00,      158.00),
    (  700,    130.00,      174.00),
    (  800,    141.00,      188.00),
    (  900,    151.00,      202.00),
    ( 1000,    160.00,      214.00),
]

# ============================================================
#  TARIFA POR PALÉ + PROVINCIA  (hasta 1000 kg por palé)
#  Aquí se listan provincias representativas; añade/modifica las que necesites
# ============================================================
TARIFA_PALE_PROVINCIA = {
    'Madrid':      28.00,
    'Barcelona':   35.00,
    'Valencia':    32.00,
    'Sevilla':     38.00,
    'Zaragoza':    30.00,
    'Bilbao':      36.00,
    'Málaga':      40.00,
    'Murcia':      34.00,
    'Valladolid':  31.00,
    'Baleares':    55.00,
    'Canarias':    70.00,
    'Peninsula_media': 34.00,   # valor medio para cálculos genéricos
}

# ============================================================
#  TARIFA MULTI-PALÉ  (a partir de 5 palés)
#  Formato: (num_pales_max, kg_max, precio_total)
# ============================================================
TRANSPORTE_MULTIPALE = [
    #  palés_max   kg_max   precio_total
    (    8,         5000,    210.00),
    (   10,         7000,    270.00),
    (   12,         9000,    320.00),
    (   15,        12000,    380.00),
    (   18,        15000,    440.00),
    (   21,        17000,    490.00),
    (   24,        19000,    535.00),
    (   27,        21000,    580.00),
    (   30,        23000,    620.00),
    (   33,        25000,    660.00),
]

# ============================================================
#  ALMACÉN CENTRAL MADRID
# ============================================================
ALMACEN_MADRID = {
    'almacenaje_m3_dia':        0.17,
    'recepcion_kg':             0.01,
    'manipulados_pedido':       0.53,
    'manipulados_unidad_bulto': 0.12,
    'gestion_admin_pedido':     0.43,
    'inventarios_hora':        24.95,
    'desmanipulados_pedido':    0.53,
    'desmanipulados_unidad':    0.12,
    'sms':                      0.17,
    'acondicionamiento_botella':0.48,
    'recogidas_unidad':         3.30,
    'duas_unidad':             40.00,
}

# ============================================================
#  ALMACENES REGIONALES  (precio por caja según volumen mensual)
# ============================================================
ALMACEN_REGIONAL_CAJAS = [
    #  cajas_max   €/caja
    (    700,      3.08),
    (   3500,      2.15),
    (   7000,      1.54),
    (  14000,      0.92),
    (  35000,      0.615),
]
ALMACEN_REGIONAL_RECEPCION_CAJA = 0.63
ALMACEN_REGIONAL_MANIPULACION_PEDIDO = 8.20
ALMACEN_REGIONAL_SEGURO_PCT = 0.0008   # 0.08%

print('Tarifas cargadas ✓')

---
## 🔧 SECCIÓN 3 — FUNCIONES DE CÁLCULO

In [ ]:
# ─── Geometría del palé ───────────────────────────────────────────────────────

def cajas_por_pale(producto: dict) -> int:
    """Calcula cuántas cajas caben en un europalet (base + altura)."""
    cajas_x = PALE_LARGO_CM // producto['largo_cm']
    cajas_y = PALE_ANCHO_CM // producto['ancho_cm']
    capas   = PALE_ALTO_MAX_CM // producto['alto_cm']
    por_geometria = max(1, cajas_x * cajas_y * capas)
    # límite por peso
    por_peso = max(1, int(PALE_PESO_MAX_KG / producto['peso_kg']))
    return min(por_geometria, por_peso)

def volumen_m3_caja(producto: dict) -> float:
    return (producto['largo_cm'] * producto['ancho_cm'] * producto['alto_cm']) / 1_000_000


# ─── Coste transporte ────────────────────────────────────────────────────────

def coste_transporte_peso(peso_kg: float, destino: str = 'peninsula') -> float:
    """Tarifa básica por peso (hasta 1000 kg)."""
    col = 1 if destino == 'peninsula' else 2
    for tramo in TRANSPORTE_PESO:
        if peso_kg <= tramo[0]:
            return tramo[col]
    return TRANSPORTE_PESO[-1][col]  # fallback al máximo


def coste_transporte_pale_unitario(provincia: str = 'Peninsula_media') -> float:
    """Tarifa por palé individual según provincia."""
    return TARIFA_PALE_PROVINCIA.get(provincia, TARIFA_PALE_PROVINCIA['Peninsula_media'])


def coste_transporte_multipale(num_pales: int, peso_total_kg: float) -> float | None:
    """Tarifa multi-palé (>=5). Devuelve None si no aplica."""
    if num_pales < 5:
        return None
    for tramo in TRANSPORTE_MULTIPALE:
        if num_pales <= tramo[0] and peso_total_kg <= tramo[1]:
            return tramo[2]
    # Si supera todos los tramos, extrapolar el último
    return TRANSPORTE_MULTIPALE[-1][2]


def mejor_tarifa_transporte(num_cajas: int, producto: dict,
                             destino: str = 'peninsula',
                             provincia: str = 'Peninsula_media') -> dict:
    """
    Calcula el coste óptimo de transporte comparando las tres modalidades
    y devuelve la más barata junto con el desglose.
    """
    peso_total = num_cajas * producto['peso_kg']
    cpale      = cajas_por_pale(producto)
    num_pales  = max(1, int(np.ceil(num_cajas / cpale)))

    opciones = {}

    # Opción 1: tarifa por peso
    if peso_total <= 1000:
        opciones['por_peso'] = coste_transporte_peso(peso_total, destino)

    # Opción 2: tarifa por palé(s) individuales
    coste_pales_ind = num_pales * coste_transporte_pale_unitario(provincia)
    opciones['pales_individuales'] = coste_pales_ind

    # Opción 3: multi-palé
    coste_mp = coste_transporte_multipale(num_pales, peso_total)
    if coste_mp is not None:
        opciones['multi_pale'] = coste_mp

    mejor = min(opciones, key=opciones.get)
    return {
        'coste_transporte': opciones[mejor],
        'modalidad': mejor,
        'num_pales': num_pales,
        'peso_total_kg': peso_total,
        'opciones': opciones,
    }


# ─── Coste almacén regional ──────────────────────────────────────────────────

def tarifa_almacen_regional_por_caja(num_cajas_mes: int) -> float:
    """Precio por caja según volumen mensual."""
    for tramo in ALMACEN_REGIONAL_CAJAS:
        if num_cajas_mes <= tramo[0]:
            return tramo[1]
    return ALMACEN_REGIONAL_CAJAS[-1][1]


def coste_almacen_regional(num_cajas: int, valor_mercancia: float,
                            num_pedidos_salida: int = 1) -> dict:
    """Coste completo de almacén regional para un envío."""
    tarifa_caja = tarifa_almacen_regional_por_caja(num_cajas)
    almacenaje  = num_cajas * tarifa_caja
    recepcion   = num_cajas * ALMACEN_REGIONAL_RECEPCION_CAJA
    manipulacion= num_pedidos_salida * ALMACEN_REGIONAL_MANIPULACION_PEDIDO
    seguro      = valor_mercancia * ALMACEN_REGIONAL_SEGURO_PCT
    total       = almacenaje + recepcion + manipulacion + seguro
    return {
        'coste_almacen_regional': total,
        'almacenaje': almacenaje,
        'recepcion': recepcion,
        'manipulacion': manipulacion,
        'seguro': seguro,
        'tarifa_caja_aplicada': tarifa_caja,
    }


# ─── Coste almacén Madrid ────────────────────────────────────────────────────

def coste_almacen_madrid(num_cajas: int, producto: dict,
                          dias_almacenaje: int = 30,
                          num_pedidos: int = 1) -> dict:
    """Coste de almacén central Madrid para un lote."""
    vol_m3      = volumen_m3_caja(producto) * num_cajas
    peso_total  = num_cajas * producto['peso_kg']
    almacenaje  = vol_m3 * dias_almacenaje * ALMACEN_MADRID['almacenaje_m3_dia']
    recepcion   = peso_total * ALMACEN_MADRID['recepcion_kg']
    manip       = (num_pedidos * ALMACEN_MADRID['manipulados_pedido'] +
                   num_cajas   * ALMACEN_MADRID['manipulados_unidad_bulto'])
    gestion     = num_pedidos  * ALMACEN_MADRID['gestion_admin_pedido']
    total       = almacenaje + recepcion + manip + gestion
    return {
        'coste_almacen_madrid': total,
        'almacenaje_m3': almacenaje,
        'recepcion': recepcion,
        'manipulacion': manip,
        'gestion': gestion,
        'volumen_m3': vol_m3,
    }

print('Funciones definidas ✓')

---
## 📊 SECCIÓN 4 — ANÁLISIS POR PRODUCTO Y ESCENARIO

In [ ]:
def analizar_producto(nombre: str, producto: dict,
                      escenarios: list,
                      destino: str = 'peninsula',
                      provincia: str = 'Peninsula_media',
                      valor_por_caja: float = 50.0,
                      dias_almacenaje_madrid: int = 30,
                      num_pedidos_salida: int = 1) -> pd.DataFrame:
    filas = []
    for n_cajas in escenarios:
        tr  = mejor_tarifa_transporte(n_cajas, producto, destino, provincia)
        alm = coste_almacen_regional(n_cajas, n_cajas * valor_por_caja, num_pedidos_salida)
        mad = coste_almacen_madrid(n_cajas, producto, dias_almacenaje_madrid, num_pedidos_salida)

        coste_total = tr['coste_transporte'] + alm['coste_almacen_regional']
        coste_x_caja = coste_total / n_cajas

        filas.append({
            'Producto': nombre,
            'Cajas': n_cajas,
            'Peso_total_kg': tr['peso_total_kg'],
            'Palés': tr['num_pales'],
            'Modalidad_transporte': tr['modalidad'],
            'Coste_transporte': tr['coste_transporte'],
            'Tarifa_caja_almacen': alm['tarifa_caja_aplicada'],
            'Coste_almacen_regional': alm['coste_almacen_regional'],
            'Coste_almacen_madrid': mad['coste_almacen_madrid'],
            'Coste_total_logistico': coste_total,
            'Coste_por_caja': coste_x_caja,
        })
    return pd.DataFrame(filas)


# ── Ejecutar análisis para los 5 productos ───────────────────────────────────
# Puedes ajustar estos parámetros por producto
PARAMETROS_COMUNES = dict(
    destino              = 'peninsula',
    provincia            = 'Peninsula_media',
    valor_por_caja       = 50.0,     # € — cambia por producto si es necesario
    dias_almacenaje_madrid = 30,
    num_pedidos_salida   = 1,
)

resultados = pd.concat([
    analizar_producto(nombre, prod, ESCENARIOS_CAJAS, **PARAMETROS_COMUNES)
    for nombre, prod in PRODUCTOS.items()
], ignore_index=True)

# Formato
cols_euro = ['Coste_transporte','Coste_almacen_regional','Coste_almacen_madrid',
             'Coste_total_logistico','Coste_por_caja','Tarifa_caja_almacen']
display_df = resultados.copy()
for c in cols_euro:
    display_df[c] = display_df[c].map('{:.2f} €'.format)

print('\n📋 TABLA COMPLETA DE RESULTADOS')
display_df

---
## 🏆 SECCIÓN 5 — PUNTOS ÓPTIMOS

In [ ]:
print('\n🏆 COSTE MÍNIMO POR CAJA — PUNTO ÓPTIMO POR PRODUCTO\n')
optimos = []
for nombre in PRODUCTOS:
    sub = resultados[resultados['Producto'] == nombre]
    idx_opt = sub['Coste_por_caja'].idxmin()
    row = sub.loc[idx_opt]
    optimos.append(row)
    print(f"  {nombre}: {int(row['Cajas'])} cajas → {row['Coste_por_caja']:.3f} €/caja "
          f"(transporte: {row['Modalidad_transporte']}, total: {row['Coste_total_logistico']:.2f} €)")

df_optimos = pd.DataFrame(optimos)[['Producto','Cajas','Modalidad_transporte',
                                     'Palés','Coste_transporte',
                                     'Coste_almacen_regional','Coste_total_logistico',
                                     'Coste_por_caja']]
print()
fmt = df_optimos.copy()
for c in ['Coste_transporte','Coste_almacen_regional','Coste_total_logistico']:
    fmt[c] = fmt[c].map('{:.2f} €'.format)
fmt['Coste_por_caja'] = fmt['Coste_por_caja'].map('{:.3f} €'.format)
print(fmt.to_string(index=False))


---
## 📈 SECCIÓN 6 — VISUALIZACIONES

In [ ]:
# ── Gráfico 1: Coste total por número de cajas (por producto) ───────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharey=False)
axes = axes.flatten()
colors = plt.cm.tab10.colors

for i, (nombre, prod) in enumerate(PRODUCTOS.items()):
    ax = axes[i]
    sub = resultados[resultados['Producto'] == nombre]
    ax.plot(sub['Cajas'], sub['Coste_total_logistico'], '-o', color=colors[i], linewidth=2, markersize=5)
    ax.set_title(nombre, fontsize=11, fontweight='bold')
    ax.set_xlabel('Cajas por envío')
    ax.set_ylabel('Coste total (€)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}€'))
    ax.grid(True, alpha=0.3)
    # Marcar óptimo
    idx_opt = sub['Coste_por_caja'].idxmin()
    ax.axvline(sub.loc[idx_opt, 'Cajas'], color='red', linestyle='--', alpha=0.7, linewidth=1.2)

axes[-1].axis('off')  # celda vacía
fig.suptitle('Coste total logístico por volumen de envío\n(línea roja = punto óptimo coste/caja)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('/sessions/serene-pensive-cori/coste_total.png', bbox_inches='tight', dpi=130)
plt.show()
print('Gráfico 1 guardado ✓')

In [ ]:
# ── Gráfico 2: Coste por caja (eficiencia logística) ────────────────────────
fig, ax = plt.subplots(figsize=(13, 6))

for i, (nombre, prod) in enumerate(PRODUCTOS.items()):
    sub = resultados[resultados['Producto'] == nombre]
    ax.plot(sub['Cajas'], sub['Coste_por_caja'], '-o', label=nombre,
            color=colors[i], linewidth=2, markersize=5)

ax.set_title('Coste logístico por caja según volumen de envío', fontsize=13, fontweight='bold')
ax.set_xlabel('Número de cajas por envío', fontsize=11)
ax.set_ylabel('€ por caja', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.2f}€'))
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/sessions/serene-pensive-cori/coste_por_caja.png', bbox_inches='tight', dpi=130)
plt.show()
print('Gráfico 2 guardado ✓')

In [ ]:
# ── Gráfico 3: Desglose transporte vs almacén en punto óptimo ───────────────
nombres = [r['Producto'] for r in optimos]
tr_costes  = [r['Coste_transporte'] for r in optimos]
alm_costes = [r['Coste_almacen_regional'] for r in optimos]

x = np.arange(len(nombres))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - width/2, tr_costes,  width, label='Transporte', color='steelblue')
b2 = ax.bar(x + width/2, alm_costes, width, label='Almacén regional', color='coral')

ax.set_title('Desglose coste en punto óptimo por producto', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(nombres, fontsize=9)
ax.set_ylabel('Coste total (€)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}€'))
ax.legend()
ax.bar_label(b1, fmt='%.0f€', padding=3, fontsize=8)
ax.bar_label(b2, fmt='%.0f€', padding=3, fontsize=8)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/sessions/serene-pensive-cori/desglose_optimo.png', bbox_inches='tight', dpi=130)
plt.show()
print('Gráfico 3 guardado ✓')

In [ ]:
# ── Gráfico 4: Mapa de calor — coste/caja por producto × escenario ──────────
pivot = resultados.pivot(index='Producto', columns='Cajas', values='Coste_por_caja')

fig, ax = plt.subplots(figsize=(13, 4))
im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn_r')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([str(c) for c in pivot.columns], fontsize=9)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=9)
ax.set_xlabel('Cajas por envío', fontsize=10)
ax.set_title('Coste logístico por caja — mapa de calor (verde = más barato)', fontsize=11, fontweight='bold')

for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f"{pivot.values[i,j]:.2f}€", ha='center', va='center', fontsize=7.5)

plt.colorbar(im, ax=ax, label='€/caja')
plt.tight_layout()
plt.savefig('/sessions/serene-pensive-cori/heatmap_coste.png', bbox_inches='tight', dpi=130)
plt.show()
print('Gráfico 4 guardado ✓')

---
## 🔍 SECCIÓN 7 — COMPARADOR MANUAL
> Introduce tus propios parámetros para evaluar un envío concreto.

In [ ]:
# ============================================================
#  MODIFICAR ESTOS PARÁMETROS PARA SIMULAR UN ENVÍO CONCRETO
# ============================================================
PRODUCTO_SEL   = 'Producto A'   # nombre del producto
CAJAS_ENVIO    = 200            # número de cajas
PROVINCIA_DEST = 'Barcelona'    # provincia destino
DESTINO_ZONA   = 'peninsula'    # 'peninsula' o 'baleares'
VALOR_CAJA     = 50.0           # valor mercancía por caja (€)
DIAS_ALMACEN   = 30             # días en almacén Madrid antes de envío
PEDIDOS_SALIDA = 3              # pedidos de salida que se van a generar
# ============================================================

prod = PRODUCTOS[PRODUCTO_SEL]
tr   = mejor_tarifa_transporte(CAJAS_ENVIO, prod, DESTINO_ZONA, PROVINCIA_DEST)
alm  = coste_almacen_regional(CAJAS_ENVIO, CAJAS_ENVIO * VALOR_CAJA, PEDIDOS_SALIDA)
mad  = coste_almacen_madrid(CAJAS_ENVIO, prod, DIAS_ALMACEN, PEDIDOS_SALIDA)

print(f"\n{'='*55}")
print(f"  SIMULACIÓN: {PRODUCTO_SEL} — {CAJAS_ENVIO} cajas → {PROVINCIA_DEST}")
print(f"{'='*55}")
print(f"  Peso total         : {tr['peso_total_kg']:>10.1f} kg")
print(f"  Palés necesarios   : {tr['num_pales']:>10}")
print(f"  Cajas por palé     : {cajas_por_pale(prod):>10}")
print(f"")
print(f"  ─── TRANSPORTE ─────────────────────────────")
for modo, precio in tr['opciones'].items():
    tag = '★' if modo == tr['modalidad'] else ' '
    print(f"  {tag} {modo:<22}: {precio:>8.2f} €")
print(f"  → Mejor opción     : {tr['modalidad']} — {tr['coste_transporte']:.2f} €")
print(f"")
print(f"  ─── ALMACÉN REGIONAL ───────────────────────")
print(f"  Tarifa/caja        : {alm['tarifa_caja_aplicada']:>10.3f} €")
print(f"  Almacenaje         : {alm['almacenaje']:>10.2f} €")
print(f"  Recepción          : {alm['recepcion']:>10.2f} €")
print(f"  Manipulación       : {alm['manipulacion']:>10.2f} €")
print(f"  Seguro             : {alm['seguro']:>10.2f} €")
print(f"  TOTAL almacén      : {alm['coste_almacen_regional']:>10.2f} €")
print(f"")
print(f"  ─── ALMACÉN MADRID (referencia) ────────────")
print(f"  Almacenaje {DIAS_ALMACEN}d       : {mad['almacenaje_m3']:>10.2f} €")
print(f"  Recepción          : {mad['recepcion']:>10.2f} €")
print(f"  Manipulación       : {mad['manipulacion']:>10.2f} €")
print(f"  Gestión            : {mad['gestion']:>10.2f} €")
print(f"  TOTAL Madrid       : {mad['coste_almacen_madrid']:>10.2f} €")
print(f"")
coste_total = tr['coste_transporte'] + alm['coste_almacen_regional']
print(f"  ═══════════════════════════════════════════")
print(f"  COSTE LOGÍSTICO TOTAL : {coste_total:>10.2f} €")
print(f"  COSTE POR CAJA        : {coste_total/CAJAS_ENVIO:>10.3f} €")
print(f"  ═══════════════════════════════════════════")

---
## 📤 SECCIÓN 8 — EXPORTAR RESULTADOS A EXCEL

In [ ]:
OUTPUT_PATH = '/sessions/serene-pensive-cori/mnt/LOGÍSTICA/analisis_logistico.xlsx'

with pd.ExcelWriter(OUTPUT_PATH, engine='openpyxl') as writer:
    resultados.to_excel(writer, sheet_name='Todos los escenarios', index=False)
    df_optimos.to_excel(writer, sheet_name='Puntos óptimos', index=False)
    pd.DataFrame(PRODUCTOS).T.reset_index().rename(columns={'index':'Producto'}).to_excel(
        writer, sheet_name='Configuración productos', index=False)

print(f'Excel exportado → {OUTPUT_PATH} ✓')